# SurviveCity Eval — Kaggle Runner

Loads a LoRA checkpoint from HF Hub and evaluates it against a random baseline.
Produces real metrics + bar plots (no fabricated curves) and pushes results back
to `<HUB_MODEL_ID>/eval_results/` so future eval runs accumulate into a
cross-checkpoint trend chart automatically.

## Setup
1. Runtime → Accelerator → **T4 GPU** (any GPU is fine, eval is fp16 inference)
2. Add-ons → Secrets → add `GITHUB_TOKEN` and `HF_TOKEN`
3. Edit `HUB_MODEL_ID` and `CHECKPOINT_TO_EVAL` in cell 1 below
4. Run All

## Runtime expectations
- **Baseline (random) eval**: ~5 min for 20 episodes
- **Trained (LLM) eval**: ~30-60 min for 5 episodes (LLM generation is the bottleneck)
- **Total wallclock**: ~30-90 min depending on `N_EPISODES_TRAINED`

If a cell errors, do **Run → Restart Kernel** then Run All again.

## 1. Configuration

In [ ]:
HUB_MODEL_ID         = "noanya/zombiee"
REPO_URL             = "https://github.com/SirjanSingh/zombiee.git"
REPO_BRANCH          = "master"

# Which checkpoint to evaluate. "latest" picks the highest-numbered checkpoint-* dir
# on the Hub. Otherwise pass a specific dir name like "checkpoint-2".
CHECKPOINT_TO_EVAL   = "latest"

# Base model — must match what training used
BASE_MODEL_NAME      = "Qwen/Qwen2.5-3B-Instruct"

# Episode counts
N_EPISODES_BASELINE  = 20      # random baseline (cheap, ~5 min)
N_EPISODES_TRAINED   = 5       # LLM inference (slow, ~30-60 min)

# Inference settings
MAX_NEW_TOKENS       = 64
TEMPERATURE          = 0.7

OUTPUT_DIR           = "/kaggle/working/eval_out"
REPO_DIR             = "/kaggle/working/zombiee"
LORA_DIR             = "/kaggle/working/lora_checkpoint"

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Kaggle dual-T4 → mask second GPU
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("WANDB_DISABLED", "true")
print(f"Eval plan: {N_EPISODES_BASELINE} baseline + {N_EPISODES_TRAINED} trained on {CHECKPOINT_TO_EVAL}")

## 2. GPU Sanity Check

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,memory.free,compute_cap --format=csv

import torch
print(f"\nPyTorch: {torch.__version__}  CUDA: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}  device count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info(0)
    print(f"GPU 0: {torch.cuda.get_device_name(0)}  {free/1e9:.1f}GB free / {total/1e9:.1f}GB total")

## 3. Install Dependencies

Lighter than training — just transformers, peft, hf_hub, matplotlib. No unsloth, no trl, no bitsandbytes (eval runs in fp16, no quantization needed).

In [ ]:
!pip install -q --upgrade pip setuptools wheel
!pip install -q --upgrade "transformers>=4.47,<4.50" "accelerate>=1.2" "peft>=0.14" "huggingface_hub>=0.25" "pydantic>=2.0" "fastapi>=0.104" "uvicorn[standard]>=0.24" "matplotlib>=3.8" hf_transfer

import torch, transformers, peft
print("torch", torch.__version__, "cuda", torch.version.cuda)
print("transformers", transformers.__version__)
print("peft", peft.__version__)
print("CUDA OK:", torch.cuda.is_available())

## 4. Clone Repo & Install Package

Same safe-clone pattern as the training notebooks: token via `http.extraheader` so it never appears in `argv` or tracebacks; errors are scrubbed before raise.

In [ ]:
import os, subprocess, base64

tok = None
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    tok = os.environ.get("GITHUB_TOKEN")
if not tok:
    import getpass
    tok = getpass.getpass("GitHub PAT: ")

subprocess.run(["rm", "-rf", REPO_DIR], check=False)

b64 = base64.b64encode(f"x-access-token:{tok}".encode()).decode()
auth = f"http.https://github.com/.extraheader=Authorization: Basic {b64}"

env = os.environ.copy()
env["GIT_TERMINAL_PROMPT"] = "0"
env["GIT_ASKPASS"] = "/bin/echo"

try:
    subprocess.run(
        ["git", "-c", auth, "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR],
        env=env, check=True, capture_output=True, text=True,
    )
except subprocess.CalledProcessError as e:
    msg = ((e.stdout or "") + (e.stderr or "")).replace(tok, "***").replace(b64, "***")
    raise RuntimeError(f"git clone failed (rc={e.returncode}):\n{msg}") from None

subprocess.run(["pip", "install", "-q", "-e", "."], cwd=REPO_DIR, check=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

log = subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-3"],
                     capture_output=True, text=True).stdout
print(log)
del tok, b64, auth
print("Repo cloned and installed")

## 5. HuggingFace Hub Login

In [ ]:
from huggingface_hub import login, HfApi

hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    client = UserSecretsClient()
    for name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            hf_token = client.get_secret(name)
            if hf_token:
                break
        except Exception:
            continue
except Exception:
    hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
if not hf_token:
    import getpass
    hf_token = getpass.getpass("HF token (write scope): ")

login(token=hf_token, add_to_git_credential=True)
os.environ["HUGGINGFACE_TOKEN"] = hf_token
os.environ["HF_TOKEN"] = hf_token

api = HfApi()
print("HF login OK")

## 6. List Available Checkpoints on Hub

In [ ]:
files = api.list_repo_files(HUB_MODEL_ID)
checkpoint_dirs = sorted(
    {f.split('/')[0] for f in files if f.startswith('checkpoint-')},
    key=lambda s: int(s.split('-')[1]) if s.split('-')[1].isdigit() else 0,
)
has_root_adapter = any(
    f in ('adapter_model.safetensors', 'adapter_model.bin', 'adapter_config.json')
    for f in files
)

if checkpoint_dirs:
    print(f"Found checkpoint-* subdirs on {HUB_MODEL_ID}:")
    for cp in checkpoint_dirs:
        print(f"  - {cp}")
elif has_root_adapter:
    print(f"No checkpoint-* subdirs found, but adapter exists at root of {HUB_MODEL_ID}.")
    print("This is normal when training uses hub_strategy='every_save' (pushes only the latest).")
else:
    raise RuntimeError(
        f"No adapter artifacts on {HUB_MODEL_ID}. "
        f"Train first (training cell needs to complete at least one save_steps cycle)."
    )

# Decide what to evaluate
if CHECKPOINT_TO_EVAL == "latest":
    if checkpoint_dirs:
        selected = checkpoint_dirs[-1]
    else:
        selected = "root"
elif CHECKPOINT_TO_EVAL == "root":
    selected = "root"
elif CHECKPOINT_TO_EVAL in checkpoint_dirs:
    selected = CHECKPOINT_TO_EVAL
else:
    raise ValueError(
        f"{CHECKPOINT_TO_EVAL!r} not on hub. Available: {checkpoint_dirs or ['root']}"
    )

# Determine the global step number (used for eval_step_NNNN.json naming)
if selected == "root":
    # Read trainer_state.json to find the current global_step
    from huggingface_hub import hf_hub_download
    try:
        ts_path = hf_hub_download(
            repo_id=HUB_MODEL_ID,
            filename='trainer_state.json',
            local_dir=OUTPUT_DIR,
        )
        ts = json.load(open(ts_path))
        SELECTED_STEP = int(ts.get('global_step', 0))
        print(f"Current step (from root trainer_state.json): {SELECTED_STEP}")
    except Exception as e:
        print(f"Warning: could not read trainer_state.json ({e}); using step 0")
        SELECTED_STEP = 0
else:
    SELECTED_STEP = int(selected.split('-')[1])

print(f"\n=> Will evaluate: {selected}  (step {SELECTED_STEP})")

## 7. Download Selected Checkpoint

In [ ]:
from huggingface_hub import snapshot_download
import shutil

shutil.rmtree(LORA_DIR, ignore_errors=True)
os.makedirs(LORA_DIR, exist_ok=True)

if selected == "root":
    print(f"Downloading root-level adapter from {HUB_MODEL_ID} ...")
    snapshot_download(
        repo_id=HUB_MODEL_ID,
        allow_patterns=[
            "adapter_model.safetensors",
            "adapter_model.bin",
            "adapter_config.json",
            "trainer_state.json",
            "tokenizer*",
            "special_tokens_map.json",
            "added_tokens.json",
            "chat_template*",
            "vocab.json",
            "merges.txt",
        ],
        local_dir=LORA_DIR,
    )
    ADAPTER_PATH = LORA_DIR
else:
    print(f"Downloading {selected}/ from {HUB_MODEL_ID} ...")
    snapshot_download(
        repo_id=HUB_MODEL_ID,
        allow_patterns=[f"{selected}/*"],
        local_dir=LORA_DIR,
    )
    ADAPTER_PATH = os.path.join(LORA_DIR, selected)

assert os.path.isdir(ADAPTER_PATH), f"Expected {ADAPTER_PATH} to exist"
print(f"Adapter at: {ADAPTER_PATH}")
print(f"Files: {sorted(os.listdir(ADAPTER_PATH))}")

## 8. Run Baseline Evaluation (Random Actions)

In [ ]:
import sys, random, time
sys.path.insert(0, REPO_DIR)
from survivecity_env.env import SurviveCityEnv

def run_baseline_episodes(n_episodes, seed=42):
    rng = random.Random(seed)
    actions = ['move_up', 'move_down', 'move_left', 'move_right', 'eat', 'wait']
    results = []
    t0 = time.time()
    for ep in range(n_episodes):
        ep_seed = rng.randint(0, 999_999)
        env = SurviveCityEnv()
        obs = env.reset(seed=ep_seed)
        total_reward = 0.0
        steps = 0
        while not obs.get('done', False) and steps < 350:
            aid = obs.get('metadata', {}).get('current_agent_id', 0)
            sc = obs.get('step_count', 0)
            if sc == 50:
                action = {'agent_id': aid, 'action_type': 'vote_lockout',
                          'vote_target': rng.choice([0, 1, 2])}
            else:
                action = {'agent_id': aid, 'action_type': rng.choice(actions)}
            obs = env.step(action)
            total_reward += obs.get('reward', 0.5)
            steps += 1
        meta = obs.get('metadata', {})
        results.append({
            'episode': ep,
            'total_reward': total_reward,
            'final_step': obs.get('step_count', 0),
            'survived': meta.get('healthy_alive', 0) >= 1,
            'vote_correct': meta.get('vote_correct'),
        })
        if (ep + 1) % 5 == 0:
            sr = sum(1 for r in results if r['survived']) / len(results)
            print(f"  baseline {ep+1}/{n_episodes}  survival={sr:.1%}  elapsed={time.time()-t0:.0f}s")
    return results

print(f"Running baseline ({N_EPISODES_BASELINE} episodes) ...")
baseline_results = run_baseline_episodes(N_EPISODES_BASELINE)
print(f"DONE: {len(baseline_results)} baseline episodes")

## 9. Load Trained Model + Run Trained Evaluation

Loads base model in **fp16** (no bnb 4-bit). Inference doesn't need quantization
and avoiding bnb dodges all the T4 + bf16 + cublas issues from training. ~6 GB
for the model + LoRA + activations fits comfortably in T4's 15 GB.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from survivecity_env.prompts import build_system_prompt

print(f"Loading base model: {BASE_MODEL_NAME} (fp16)")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
)

print(f"Loading LoRA adapter: {ADAPTER_PATH}")
model = PeftModel.from_pretrained(base, ADAPTER_PATH)
model.eval()

free, total = torch.cuda.mem_get_info(0)
print(f"GPU memory after load: {free/1e9:.1f}GB free / {total/1e9:.1f}GB total")

import json as _json

VALID_ACTIONS = {'move_up', 'move_down', 'move_left', 'move_right',
                 'eat', 'wait', 'vote_lockout', 'broadcast'}

def parse_action(text, agent_id):
    text = text.strip()
    if text.startswith('```'):
        parts = text.split('```')
        if len(parts) >= 2:
            text = parts[1].lstrip('json').strip()
    for s in range(len(text)):
        if text[s] == '{':
            for e in range(len(text), s, -1):
                if text[e-1] == '}':
                    try:
                        d = _json.loads(text[s:e])
                        if d.get('action_type') in VALID_ACTIONS:
                            d['agent_id'] = agent_id
                            return d
                    except (_json.JSONDecodeError, TypeError):
                        continue
    return None

parse_failures = {'count': 0, 'total': 0}

def llm_action(agent_id, obs):
    desc = obs.get('description', '')
    prompt = build_system_prompt(agent_id, desc)
    messages = [
        {'role': 'system', 'content': prompt},
        {'role': 'user', 'content': 'What is your next action? Respond with JSON only.'},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True,
    )
    parse_failures['total'] += 1
    parsed = parse_action(response, agent_id)
    if parsed is None:
        parse_failures['count'] += 1
        sc = obs.get('step_count', 0)
        if sc == 50:
            return {'agent_id': agent_id, 'action_type': 'vote_lockout',
                    'vote_target': random.choice([0, 1, 2])}
        return {'agent_id': agent_id,
                'action_type': random.choice(['move_up', 'move_down',
                                              'move_left', 'move_right',
                                              'eat', 'wait'])}
    return parsed

def run_trained_episodes(n_episodes, seed=43):
    rng = random.Random(seed)
    results = []
    for ep in range(n_episodes):
        ep_seed = rng.randint(0, 999_999)
        env = SurviveCityEnv()
        obs = env.reset(seed=ep_seed)
        total_reward = 0.0
        steps = 0
        t_start = time.time()
        while not obs.get('done', False) and steps < 350:
            aid = obs.get('metadata', {}).get('current_agent_id', 0)
            action = llm_action(aid, obs)
            obs = env.step(action)
            total_reward += obs.get('reward', 0.5)
            steps += 1
        meta = obs.get('metadata', {})
        elapsed = time.time() - t_start
        results.append({
            'episode': ep,
            'total_reward': total_reward,
            'final_step': obs.get('step_count', 0),
            'survived': meta.get('healthy_alive', 0) >= 1,
            'vote_correct': meta.get('vote_correct'),
            'time_sec': elapsed,
        })
        sr = sum(1 for r in results if r['survived']) / len(results)
        print(f"  trained {ep+1}/{n_episodes}  survival={sr:.1%}  ep_time={elapsed:.0f}s")
    return results

print(f"\nRunning trained eval ({N_EPISODES_TRAINED} episodes) — this is the slow part ...")
trained_results = run_trained_episodes(N_EPISODES_TRAINED)
print(f"DONE: {len(trained_results)} trained episodes")
print(f"Action parse failures: {parse_failures['count']}/{parse_failures['total']} "
      f"({parse_failures['count']/max(1,parse_failures['total']):.1%})")

## 10. Compute Metrics & Generate Plots

In [ ]:
import json, numpy as np, datetime
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.makedirs(OUTPUT_DIR, exist_ok=True)

def compute_metrics(results, label):
    n = len(results)
    survived = sum(1 for r in results if r['survived'])
    vote_correct = sum(1 for r in results if r['vote_correct'] is True)
    vote_total = sum(1 for r in results if r['vote_correct'] is not None)
    rewards = [r['total_reward'] for r in results]
    lengths = [r['final_step'] for r in results]
    return {
        'label': label,
        'n_episodes': n,
        'survival_rate': survived / max(1, n),
        'survival_count': survived,
        'vote_accuracy': vote_correct / max(1, vote_total),
        'vote_correct_count': vote_correct,
        'vote_total_count': vote_total,
        'mean_total_reward': float(np.mean(rewards)) if rewards else 0.0,
        'std_total_reward': float(np.std(rewards)) if rewards else 0.0,
        'mean_episode_length': float(np.mean(lengths)) if lengths else 0.0,
        'std_episode_length': float(np.std(lengths)) if lengths else 0.0,
    }

baseline_m = compute_metrics(baseline_results, 'baseline')
trained_m = compute_metrics(trained_results, 'trained')

print('=' * 64)
print(f"{'Metric':<22} {'Baseline':>14} {'Trained':>14} {'Δ':>10}")
print('=' * 64)
def row(name, b, t, fmt='.2%'):
    delta = t - b
    print(f"{name:<22} {format(b, fmt):>14} {format(t, fmt):>14} "
          f"{('+' if delta >= 0 else ''):<1}{format(delta, fmt):>9}")
row('Survival rate', baseline_m['survival_rate'], trained_m['survival_rate'])
row('Vote accuracy', baseline_m['vote_accuracy'], trained_m['vote_accuracy'])
row('Mean total reward',
    baseline_m['mean_total_reward'], trained_m['mean_total_reward'], '.3f')
row('Mean episode length',
    baseline_m['mean_episode_length'], trained_m['mean_episode_length'], '.1f')
print('=' * 64)

# Bar plots
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
labels = ['Baseline', 'Trained']
colors = ['#ff9900', '#4169e1']

axes[0].bar(labels, [baseline_m['survival_rate'], trained_m['survival_rate']],
            color=colors)
axes[0].set_ylim(0, 1)
axes[0].set_ylabel('Fraction')
axes[0].set_title(f"Survival Rate — {selected}")
for i, v in enumerate([baseline_m['survival_rate'], trained_m['survival_rate']]):
    axes[0].text(i, v + 0.01, f'{v:.1%}', ha='center')

axes[1].bar(labels, [baseline_m['vote_accuracy'], trained_m['vote_accuracy']],
            color=colors)
axes[1].set_ylim(0, 1)
axes[1].set_ylabel('Fraction')
axes[1].set_title(f"Vote Accuracy — {selected}")
for i, v in enumerate([baseline_m['vote_accuracy'], trained_m['vote_accuracy']]):
    axes[1].text(i, v + 0.01, f'{v:.1%}', ha='center')

axes[2].bar(labels,
            [baseline_m['mean_total_reward'], trained_m['mean_total_reward']],
            yerr=[baseline_m['std_total_reward'], trained_m['std_total_reward']],
            color=colors, capsize=5)
axes[2].set_ylabel('Total Reward')
axes[2].set_title(f"Mean Total Reward — {selected}")

fig.tight_layout()
bar_path = f"{OUTPUT_DIR}/eval_step_{SELECTED_STEP:04d}_bars.png"
fig.savefig(bar_path, dpi=120, bbox_inches='tight')
plt.close()
print(f"\nSaved plot: {bar_path}")

# Save metrics JSON
results_json = {
    'checkpoint': selected,
    'step': SELECTED_STEP,
    'hub_model_id': HUB_MODEL_ID,
    'base_model': BASE_MODEL_NAME,
    'n_baseline': N_EPISODES_BASELINE,
    'n_trained': N_EPISODES_TRAINED,
    'baseline': baseline_m,
    'trained': trained_m,
    'baseline_episodes': baseline_results,
    'trained_episodes': trained_results,
    'parse_failure_rate': (
        parse_failures['count'] / max(1, parse_failures['total'])
    ),
    'timestamp_utc': datetime.datetime.utcnow().isoformat() + 'Z',
}
results_path = f"{OUTPUT_DIR}/eval_step_{SELECTED_STEP:04d}.json"
with open(results_path, 'w') as f:
    json.dump(results_json, f, indent=2, default=str)
print(f"Saved metrics: {results_path}")

## 11. Push Eval Results to Hub for Cross-Checkpoint Comparison

Uploads the JSON + bar PNG to `<HUB_MODEL_ID>/eval_results/`. Each new eval
adds a file there, so cell 12 below can plot a real trend across all evaluated
checkpoints (no fabrication).

In [ ]:
api.upload_file(
    path_or_fileobj=results_path,
    path_in_repo=f"eval_results/eval_step_{SELECTED_STEP:04d}.json",
    repo_id=HUB_MODEL_ID,
    commit_message=(
        f"eval: step {SELECTED_STEP} ({selected}) — survival "
        f"{trained_m['survival_rate']:.1%} (baseline "
        f"{baseline_m['survival_rate']:.1%})"
    ),
)
api.upload_file(
    path_or_fileobj=bar_path,
    path_in_repo=f"eval_results/eval_step_{SELECTED_STEP:04d}_bars.png",
    repo_id=HUB_MODEL_ID,
    commit_message=f"eval bars step {SELECTED_STEP}",
)
print(f"Pushed: https://huggingface.co/{HUB_MODEL_ID}/tree/main/eval_results")

## 12. Historical Comparison Across All Evaluated Checkpoints

Pulls every `eval_step_*.json` from the Hub and plots survival / vote-accuracy /
mean-reward as a function of training step. After you've evaluated 2+ checkpoints
this gives you a real (non-synthetic) trend chart for the demo + README.

In [ ]:
from huggingface_hub import hf_hub_download

all_files = api.list_repo_files(HUB_MODEL_ID)
result_files = sorted(
    f for f in all_files
    if f.startswith('eval_results/eval_step_') and f.endswith('.json')
)

print(f"Found {len(result_files)} eval result file(s) on Hub")

if len(result_files) < 2:
    print("Need >=2 checkpoints evaluated to plot a trend. Re-run this notebook "
          "with a different CHECKPOINT_TO_EVAL to add more points.")
else:
    history = []
    for rf in result_files:
        local = hf_hub_download(repo_id=HUB_MODEL_ID, filename=rf,
                                local_dir=OUTPUT_DIR)
        with open(local) as f:
            history.append(json.load(f))
    history.sort(key=lambda d: d['step'])

    steps    = [d['step'] for d in history]
    bl_surv  = [d['baseline']['survival_rate'] for d in history]
    tr_surv  = [d['trained']['survival_rate']  for d in history]
    bl_vote  = [d['baseline']['vote_accuracy'] for d in history]
    tr_vote  = [d['trained']['vote_accuracy']  for d in history]
    bl_rew   = [d['baseline']['mean_total_reward'] for d in history]
    tr_rew   = [d['trained']['mean_total_reward']  for d in history]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    for ax, bl, tr, ylabel, title in [
        (axes[0], bl_surv, tr_surv, 'Survival Rate', 'Survival Rate vs Training Step'),
        (axes[1], bl_vote, tr_vote, 'Vote Accuracy', 'Vote Accuracy vs Training Step'),
        (axes[2], bl_rew,  tr_rew,  'Mean Total Reward', 'Mean Reward vs Training Step'),
    ]:
        ax.plot(steps, bl, 'o--', color='#ff9900', label='Baseline')
        ax.plot(steps, tr, 'o-',  color='#4169e1', label='Trained')
        ax.set_xlabel('Training Step')
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.legend()
        ax.grid(alpha=0.3)
        if 'Reward' not in ylabel:
            ax.set_ylim(0, 1)
    fig.tight_layout()
    history_path = f"{OUTPUT_DIR}/eval_history.png"
    fig.savefig(history_path, dpi=120, bbox_inches='tight')
    plt.close()
    print(f"Saved: {history_path}")

    api.upload_file(
        path_or_fileobj=history_path,
        path_in_repo='eval_results/eval_history.png',
        repo_id=HUB_MODEL_ID,
        commit_message=f'eval history across {len(history)} checkpoints',
    )
    print(f"Pushed: https://huggingface.co/{HUB_MODEL_ID}/blob/main/eval_results/eval_history.png")

    print(f"\nSummary across {len(history)} checkpoints:")
    print('  step | baseline_surv | trained_surv | trained_vote | mean_reward')
    print('  -----+---------------+--------------+--------------+-------------')
    for d in history:
        s, t = d['step'], d['trained']
        b = d['baseline']
        print(f"  {s:>4} |   {b['survival_rate']:>9.1%}   |  "
              f"{t['survival_rate']:>9.1%}   |  {t['vote_accuracy']:>9.1%}   |  "
              f"{t['mean_total_reward']:>8.3f}")